In [ ]:
!pip install -q numpy datasets tiktoken huggingface_hub

exec(open("/content/colab_owt_drive_twocell.py").read())
colab_cell1()

Colab: colab_cell1() on CPU, then colab_cell2() on GPU.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
OUT_DIR (use for training): /content/drive/MyDrive/owt_chinchilla_runs
[data] tokenizing OpenWebText on CPU (~25–40 min for 505M tokens)…


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

  10M / 505M (2.0%, 0.2 min)
  20M / 505M (4.0%, 0.4 min)
  30M / 505M (5.9%, 0.5 min)
  40M / 505M (7.9%, 0.7 min)
  50M / 505M (9.9%, 0.8 min)
  60M / 505M (11.9%, 0.9 min)
  70M / 505M (13.9%, 1.1 min)
  80M / 505M (15.8%, 1.2 min)
  90M / 505M (17.8%, 1.4 min)
  100M / 505M (19.8%, 1.5 min)
  110M / 505M (21.8%, 1.7 min)
  120M / 505M (23.8%, 1.9 min)
  130M / 505M (25.7%, 2.0 min)
  140M / 505M (27.7%, 2.1 min)
  150M / 505M (29.7%, 2.3 min)
  160M / 505M (31.7%, 2.4 min)
  170M / 505M (33.7%, 2.6 min)
  180M / 505M (35.6%, 2.7 min)
  190M / 505M (37.6%, 2.8 min)
  200M / 505M (39.6%, 3.0 min)
  210M / 505M (41.6%, 3.1 min)
  220M / 505M (43.6%, 3.3 min)
  230M / 505M (45.5%, 3.4 min)
  240M / 505M (47.5%, 3.6 min)
  250M / 505M (49.5%, 3.8 min)
  260M / 505M (51.5%, 3.9 min)
  270M / 505M (53.5%, 4.0 min)
  280M / 505M (55.4%, 4.2 min)
  290M / 505M (57.4%, 4.3 min)
  300M / 505M (59.4%, 4.5 min)
  310M / 505M (61.4%, 4.6 min)
  320M / 505M (63.4%, 4.7 min)
  330M / 505M (65.3%, 

In [ ]:
import os, shutil, runpy

src = "/content/drive/MyDrive/owt_chinchilla_runs"
dst = "/content/owt_chinchilla_local"
os.makedirs(dst, exist_ok=True)
for f in ("owt_train_tokens.npy", "owt_val_tokens.npy"):
    shutil.copy2(f"{src}/{f}", f"{dst}/{f}")

os.environ["OWT_CHINCHILLA_DIR"] = src
os.environ["OWT_TOKEN_CACHE_DIR"] = dst

In [ ]:
runpy.run_path("/content/owt_chinchilla_e.py", run_name="__main__")

OUT_DIR:  /content/drive/MyDrive/owt_chinchilla_runs
  (all outputs here: token cache, per-epoch *.json, *_final_backup.json,
   manifest.json, triangulation.txt, triangulation_three.json)
TRAIN:    500M tok/epoch × 6 epochs/model
DEVICE:   cuda  |  DTYPE: torch.bfloat16
GPU:      NVIDIA A100-SXM4-40GB  |  VRAM 39.5 GiB
VOCAB:    50257  SEQ_LEN: 1024
DataLoader workers: 0  (set OWT_DATALOADER_WORKERS to override)

[data] loaded cached tokens (no HuggingFace download):
       /content/drive/MyDrive/owt_chinchilla_runs/owt_train_tokens.npy
       /content/drive/MyDrive/owt_chinchilla_runs/owt_val_tokens.npy
       train=500,000,000  val=5,000,000

TRAINING A_10M
  params: 10,165,681
  tokens/step=65536  steps/epoch=7629  total=45774
  step    200  ep1  ce=7.2108  elapsed=1.0min
  step    400  ep1  ce=6.4508  elapsed=2.1min
  step    600  ep1  ce=6.0322  elapsed=3.1min
  step    800  ep1  ce=5.7885  elapsed=4.1min
  step   1000  ep1  ce=5.6107  elapsed=5.1min
  step   1200  ep1  ce=5.4690

{'__name__': '__main__',
 '__doc__': '\n================================================================================\nChinchilla-E estimate from three small GPT-2–tokenized models on OpenWebText.\n================================================================================\n\nDefault = full Chinchilla-E protocol (matches WT103-style analysis):\n\n    TOKENS_PER_EPOCH = 500_000_000\n    N_EPOCHS         = 6\n\nso each model yields enough (T, C*) points to fit C*(T) = E_app + B·T^{-β} (the\nh8 tail uses steps ≥ one epoch, so you need several epochs — not 3).\n\n    pip install torch transformers datasets tiktoken numpy scipy\n    python owt_chinchilla_e.py\n\nMicro-batches are sized for a single A100 40GB (65536 tokens/step via batch×accum).\n\nPrepare token cache on CPU so the GPU session only trains (same folder for both):\n\n    python owt_chinchilla_e.py --prepare-data\n\nWhat it does:\n1) Streams OpenWebText, tokenizes with GPT-2 BPE, caches train+val tokens.\n2) Trains thre

In [ ]:
from google.colab import runtime
runtime.unassign()